In [4]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [3]:
df = pd.read_csv("ALL_PROJECT.csv") #117 y 56

In [5]:
df.head()

,Time,ImageIndex,ImageName,X,Y,Z,participante
0,7.984,0,3,0.0052,1.0046,1.9762,1
1,8.017,0,3,0.0052,1.0060,1.9764,1
2,8.052,0,3,0.0053,1.0070,1.9766,1
3,8.084,0,3,0.0054,1.0070,1.9768,1
4,8.117,0,3,0.0056,1.0059,1.9766,1


In [2]:
!python  --version

Python 3.12.7


In [ ]:
#117, 56

In [7]:
df[(df["ImageName"] == 56)]

,Time,ImageIndex,ImageName,X,Y,Z,participante
13137,347.518,17,56,-0.1010,1.2045,2.0154,1
13138,347.534,17,56,-0.1010,1.2044,2.0154,1
13139,347.551,17,56,-0.1010,1.2043,2.0154,1
13140,347.568,17,56,-0.1009,1.2041,2.0153,1
13141,347.584,17,56,-0.1008,1.2040,2.0153,1
...,...,...,...,...,...,...,...
857095,443.471,21,56,0.1099,1.2050,2.0177,30
857096,443.503,21,56,0.1088,1.2048,2.0176,30
857097,443.537,21,56,0.1074,1.2046,2.0176,30
857098,443.570,21,56,0.1059,1.2045,2.0175,30


In [11]:
df[(df["ImageName"] == 57)]

,Time,ImageIndex,ImageName,X,Y,Z,participante
13992,367.552,18,57,-0.0559,1.1755,2.0097,1
13993,367.568,18,57,-0.0540,1.1762,2.0098,1
13994,367.584,18,57,-0.0521,1.1770,2.0100,1
13995,367.601,18,57,-0.0504,1.1777,2.0101,1
13996,367.618,18,57,-0.0486,1.1783,2.0102,1
...,...,...,...,...,...,...,...
857545,463.506,22,57,0.1000,1.0866,1.9935,30
857546,463.538,22,57,0.1025,1.0868,1.9936,30
857547,463.574,22,57,0.1051,1.0874,1.9937,30
857548,463.604,22,57,0.1079,1.0883,1.9939,30


In [39]:
PROJECT.shape

(869600, 7)

In [13]:
## PRUEBA 

In [37]:
PROJECT = pd.read_csv("ALL_PROJECT.csv")

In [21]:
import pandas as pd
import joblib
import numpy as np
import os

# 1) Definir rutas y parámetros globales
ruta_all_project = "ALL_PROJECT.csv"
carpeta_seg_pkl   = "images/datos_seg"  # Carpeta con archivos .pkl
canvas_pos   = (0.0, 1.1)
canvas_size  = (0.8, 0.6)
image_pixels = (800, 600)  # Tamaño original de las imágenes
pkl_shape = (300, 400)     # Tamaño de los arrays en pickle

# 2) Leer todo el ALL_PROJECT.csv
df_all = pd.read_csv(ruta_all_project)

# 3) Lista para acumular cada DataFrame procesado
lista_dfs = []

def procesar_un_subconjunto_pickle(df_eye, participante, image_name, carpeta_segmentaciones_pkl, 
                                 canvas_pos, canvas_size, image_pixels, pkl_shape):
    """
    Procesa un subconjunto de datos de eye-tracking para un participante y una imagen específica,
    usando archivos pickle con arrays de segmentación.
    
    Args:
        df_eye: DataFrame completo con datos de eye-tracking
        participante: ID del participante
        image_name: Nombre/ID de la imagen
        carpeta_segmentaciones_pkl: Ruta de la carpeta con archivos .pkl
        canvas_pos: Posición del canvas (x, y)
        canvas_size: Tamaño del canvas (width, height)
        image_pixels: Tamaño de la imagen original (width, height)
        pkl_shape: Tamaño del array en pickle (height, width)
    
    Returns:
        DataFrame con las columnas adicionales de segmentación
    """
    
    # Filtrar datos para este participante e imagen
    df_sub = df_eye[(df_eye["participante"] == participante) & 
                    (df_eye["ImageName"] == image_name)].copy()
    
    if df_sub.empty:
        return pd.DataFrame()
    
    # Construir la ruta del archivo pickle
    archivo_pkl = os.path.join(carpeta_segmentaciones_pkl, f"{image_name}.pkl")
    
    # Verificar si existe el archivo
    if not os.path.exists(archivo_pkl):
        print(f"Advertencia: No se encontró {archivo_pkl}")
        return pd.DataFrame()
    
    try:
        # Cargar el array de segmentación
        segmentation_array = joblib.load(archivo_pkl)
        
        # Verificar que sea un array 2D
        if not isinstance(segmentation_array, np.ndarray) or segmentation_array.ndim != 2:
            print(f"Advertencia: {archivo_pkl} no contiene un array 2D válido")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"Error al cargar {archivo_pkl}: {e}")
        return pd.DataFrame()
    
    # Convertir coordenadas de eye-tracking a coordenadas locales del canvas
    canvas_x, canvas_y = canvas_pos
    canvas_w, canvas_h = canvas_size
    
    # Coordenadas locales dentro del canvas (0-1)
    df_sub["localX"] = (df_sub["X"] - canvas_x) / canvas_w
    df_sub["localY"] = (df_sub["Y"] - canvas_y) / canvas_h
    
    # Filtrar solo puntos que están dentro del canvas (0-1)
    mask_validos = (df_sub["localX"] >= 0) & (df_sub["localX"] <= 1) & \
                   (df_sub["localY"] >= 0) & (df_sub["localY"] <= 1)
    df_sub = df_sub[mask_validos].copy()
    
    if df_sub.empty:
        return pd.DataFrame()
    
    # Convertir a coordenadas de píxeles de la imagen original (enteros)
    img_w, img_h = image_pixels
    df_sub["pixelX"] = (df_sub["localX"] * img_w).astype(int)
    df_sub["pixelY"] = (df_sub["localY"] * img_h).astype(int)
    
    # Asegurar que estén dentro de los límites de la imagen original
    df_sub["pixelX"] = np.clip(df_sub["pixelX"], 0, img_w - 1)
    df_sub["pixelY"] = np.clip(df_sub["pixelY"], 0, img_h - 1)
    
    # Escalar a las coordenadas del array de segmentación (pkl_shape es height, width)
    pkl_h, pkl_w = pkl_shape  # (300, 400)
    df_sub["pkl_pixelX"] = (df_sub["pixelX"] * pkl_w / img_w).astype(int)  # Escalar de 800 a 400
    df_sub["pkl_pixelY"] = (df_sub["pixelY"] * pkl_h / img_h).astype(int)  # Escalar de 600 a 300

    
    # Asegurar que las coordenadas estén dentro de los límites del array pickle
    df_sub["pkl_pixelX"] = np.clip(df_sub["pkl_pixelX"], 0, pkl_w - 1)  # 0 a 399
    df_sub["pkl_pixelY"] = np.clip(df_sub["pkl_pixelY"], 0, pkl_h - 1)  # 0 a 299
    
    # Extraer las clases de segmentación para cada punto
    class_ids = []
    for _, row in df_sub.iterrows():
        x_pkl = int(row["pkl_pixelX"])  # Coordenada X en el array pickle (0-399)
        y_pkl = int(row["pkl_pixelY"])  # Coordenada Y en el array pickle (0-299)
        class_id = segmentation_array[y_pkl, x_pkl]  # array[fila, columna]
        class_ids.append(class_id)

    df_sub["class_id"] = class_ids
    
    # Opcional: Agregar información adicional sobre las clases
    unique_classes = np.unique(segmentation_array)
    df_sub["total_classes_in_image"] = len(unique_classes)
    
    # Calcular ratio/proporción de cada clase en la imagen (opcional)
    class_ratios = []
    for class_id in df_sub["class_id"]:
        total_pixels = segmentation_array.size
        class_pixels = np.sum(segmentation_array == class_id)
        ratio = class_pixels / total_pixels
        class_ratios.append(ratio)
    
    df_sub["class_ratio"] = class_ratios
    
    return df_sub

# 4) Iterar por cada participante único
for participante in df_all["participante"].unique():
    print(f"Procesando participante: {participante}")
    
    # Filtrar las filas de ese participante
    df_part = df_all[df_all["participante"] == participante]
    
    # 5) Para cada ImageName dentro de este participante:
    for image_name in df_part["ImageName"].unique():
        print(f"  Procesando imagen: {image_name}")
        
        # 5.1) Llamar a la función para procesar ese bloque
        df_sub = procesar_un_subconjunto_pickle(
            df_eye=df_all,
            participante=participante,
            image_name=image_name,
            carpeta_segmentaciones_pkl=carpeta_seg_pkl,
            canvas_pos=canvas_pos,
            canvas_size=canvas_size,
            image_pixels=image_pixels,
            pkl_shape=pkl_shape
        )
        
        # 5.2) Si df_sub no está vacío lo guardamos en la lista
        if not df_sub.empty:
            lista_dfs.append(df_sub)
            print(f"    Procesadas {len(df_sub)} filas")

# 6) Concatenar todos los DataFrames parciales
if lista_dfs:
    df_todo_con_clase = pd.concat(lista_dfs, ignore_index=True)
    print(f"\nDataFrame final: {len(df_todo_con_clase)} filas")
    print("Columnas:", list(df_todo_con_clase.columns))
    print("Clases únicas encontradas:", sorted(df_todo_con_clase["class_id"].unique()))
else:
    # Si no se procesó nada, devolvemos un DataFrame vacío con las columnas esperadas
    columnas_finales = list(df_all.columns) + ["localX", "localY", "pixelX", "pixelY", 
                                                "class_id", 
                                               "total_classes_in_image", "class_ratio"]
    df_todo_con_clase = pd.DataFrame(columns=columnas_finales)
    print("No se procesaron datos")

# # Ejemplo de análisis de resultados
# if not df_todo_con_clase.empty:
#     print("\n=== RESUMEN DE RESULTADOS ===")
#     print(f"Total de puntos procesados: {len(df_todo_con_clase)}")
#     print(f"Participantes: {df_todo_con_clase['participante'].nunique()}")
#     print(f"Imágenes: {df_todo_con_clase['ImageName'].nunique()}")
    
#     print("\nDistribución de clases:")
#     class_counts = df_todo_con_clase["class_id"].value_counts().sort_index()
#     for class_id, count in class_counts.items():
#         percentage = (count / len(df_todo_con_clase)) * 100
#         print(f"  Clase {class_id}: {count} puntos ({percentage:.2f}%)")

Procesando participante: 1
  Procesando imagen: 3
    Procesadas 27 filas
  Procesando imagen: 4
    Procesadas 70 filas
  Procesando imagen: 5
  Procesando imagen: 7
    Procesadas 33 filas
  Procesando imagen: 8
    Procesadas 105 filas
  Procesando imagen: 9
    Procesadas 63 filas
  Procesando imagen: 30
  Procesando imagen: 32
    Procesadas 88 filas
  Procesando imagen: 33
  Procesando imagen: 36
    Procesadas 304 filas
  Procesando imagen: 39
    Procesadas 35 filas
  Procesando imagen: 41
    Procesadas 2 filas
  Procesando imagen: 43
    Procesadas 45 filas
  Procesando imagen: 47
    Procesadas 312 filas
  Procesando imagen: 48
    Procesadas 118 filas
  Procesando imagen: 50
    Procesadas 198 filas
  Procesando imagen: 51
    Procesadas 159 filas
  Procesando imagen: 56
    Procesadas 82 filas
  Procesando imagen: 57
    Procesadas 275 filas
  Procesando imagen: 58
    Procesadas 155 filas
  Procesando imagen: 63
    Procesadas 374 filas
  Procesando imagen: 66
    Procesa

In [25]:
df = df_todo_con_clase

In [43]:
print(PROJECT.shape, df.shape )

(869600, 7) (127876, 16)


In [81]:
df[(df["ImageName"] == 1)& (df["ImageIndex"] == 0) & (df["Time"]== 9.224)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio
24292,9.224,0,1,0.0307,1.1099,1.997,5,0.038375,0.0165,30,9,15,4,3,7,0.205133


In [73]:
df[(df["ImageName"] == 117)& (df["ImageIndex"] == 41)& (df["Time"] > 833)& (df["Time"] <834)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio
9283,833.180,41,117,0.2360,1.1466,2.0012,2,0.295000,0.077667,236,46,118,23,3,9,0.292692
9284,833.212,41,117,0.2360,1.1459,2.0010,2,0.295000,0.076500,236,45,118,22,3,9,0.292692
9285,833.246,41,117,0.2358,1.1447,2.0006,2,0.294750,0.074500,235,44,117,22,3,9,0.292692
9286,833.278,41,117,0.2310,1.1440,2.0005,2,0.288750,0.073333,231,43,115,21,3,9,0.292692
9287,833.311,41,117,0.2213,1.1426,2.0002,2,0.276625,0.071000,221,42,110,21,3,9,0.292692
9288,833.348,41,117,0.2116,1.1399,1.9995,2,0.264500,0.066500,211,39,105,19,3,9,0.292692
9289,833.381,41,117,0.2019,1.1361,1.9987,2,0.252375,0.060167,201,36,100,18,3,9,0.292692
9290,833.412,41,117,0.1923,1.1318,1.9977,2,0.240375,0.053000,192,31,96,15,3,9,0.292692
9291,833.446,41,117,0.1829,1.1264,1.9965,2,0.228625,0.044000,182,26,91,13,3,9,0.292692
9292,833.480,41,117,0.1738,1.1227,1.9958,2,0.217250,0.037833,173,22,86,11,3,9,0.292692


In [31]:
df[(df["ImageName"] == 56) & (df["ImageIndex"] == 17) & (df["participante"] == 1)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio
1559,348.168,17,56,0.0007,1.1330,2.0014,1,0.000875,0.055000,0,32,0,16,3,10,0.258492
1560,348.185,17,56,0.0061,1.1287,2.0005,1,0.007625,0.047833,6,28,3,14,5,10,0.317442
1561,352.734,17,56,0.2044,1.1003,1.9948,1,0.255500,0.000500,204,0,102,0,3,10,0.258492
1562,352.751,17,56,0.2064,1.1018,1.9951,1,0.258000,0.003000,206,1,103,0,3,10,0.258492
1563,352.768,17,56,0.2087,1.1021,1.9952,1,0.260875,0.003500,208,2,104,1,3,10,0.258492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1636,356.668,17,56,0.1686,1.1012,1.9952,1,0.210750,0.002000,168,1,84,0,3,10,0.258492
1637,356.685,17,56,0.1686,1.1009,1.9951,1,0.210750,0.001500,168,0,84,0,3,10,0.258492
1638,356.703,17,56,0.1686,1.1006,1.9951,1,0.210750,0.001000,168,0,84,0,3,10,0.258492
1639,356.717,17,56,0.1686,1.1004,1.9950,1,0.210750,0.000667,168,0,84,0,3,10,0.258492


In [35]:
df.shape

(127876, 16)

In [55]:
import pandas as pd
import joblib
import numpy as np
import os

def procesar_un_subconjunto_pickle_completo(df_eye, participante, image_name, carpeta_segmentaciones_pkl, 
                                          canvas_pos, canvas_size, image_pixels, pkl_shape):
    """
    Procesa un subconjunto de datos de eye-tracking MANTENIENDO TODAS LAS FILAS,
    incluso las que están fuera del canvas o tienen errores.
    """
    
    # Filtrar datos para este participante e imagen
    df_sub = df_eye[(df_eye["participante"] == participante) & 
                    (df_eye["ImageName"] == image_name)].copy()
    
    if df_sub.empty:
        return pd.DataFrame()
    
    # Inicializar nuevas columnas con valores por defecto
    df_sub["localX"] = np.nan
    df_sub["localY"] = np.nan
    df_sub["pixelX"] = np.nan
    df_sub["pixelY"] = np.nan
    df_sub["pkl_pixelX"] = np.nan
    df_sub["pkl_pixelY"] = np.nan
    df_sub["class_id"] = np.nan
    df_sub["total_classes_in_image"] = np.nan
    df_sub["class_ratio"] = np.nan
    df_sub["processing_status"] = "error"  # Para debug
    
    # Construir la ruta del archivo pickle
    archivo_pkl = os.path.join(carpeta_segmentaciones_pkl, f"{image_name}.pkl")
    
    # Verificar si existe el archivo
    if not os.path.exists(archivo_pkl):
        print(f"Advertencia: No se encontró {archivo_pkl}")
        df_sub["processing_status"] = "archivo_no_encontrado"
        return df_sub
    
    try:
        # Cargar el array de segmentación
        segmentation_array = joblib.load(archivo_pkl)
        
        # Verificar que sea un array 2D
        if not isinstance(segmentation_array, np.ndarray) or segmentation_array.ndim != 2:
            print(f"Advertencia: {archivo_pkl} no contiene un array 2D válido")
            df_sub["processing_status"] = "array_invalido"
            return df_sub
            
    except Exception as e:
        print(f"Error al cargar {archivo_pkl}: {e}")
        df_sub["processing_status"] = "error_carga"
        return df_sub
    
    # Convertir coordenadas de eye-tracking a coordenadas locales del canvas
    canvas_x, canvas_y = canvas_pos
    canvas_w, canvas_h = canvas_size
    
    # Coordenadas locales dentro del canvas (0-1) - PARA TODAS LAS FILAS
    df_sub["localX"] = (df_sub["X"] - canvas_x) / canvas_w
    df_sub["localY"] = (df_sub["Y"] - canvas_y) / canvas_h
    
    # Convertir a coordenadas de píxeles de la imagen original
    img_w, img_h = image_pixels
    
    # Calcular pixelX y pixelY, manejando NaN e infinitos
    pixel_x_float = df_sub["localX"] * img_w
    pixel_y_float = df_sub["localY"] * img_h
    
    # Reemplazar infinitos y valores muy grandes/pequeños con NaN
    pixel_x_float = pixel_x_float.replace([np.inf, -np.inf], np.nan)
    pixel_y_float = pixel_y_float.replace([np.inf, -np.inf], np.nan)
    
    # Convertir a enteros, manteniendo NaN donde corresponde
    df_sub["pixelX"] = pixel_x_float.round().astype('Int64')  # Int64 permite NaN
    df_sub["pixelY"] = pixel_y_float.round().astype('Int64')
    
    # FORZAR que pixelX y pixelY estén en rangos válidos (0 a img_w-1, 0 a img_h-1)
    # Solo para valores que no son NaN
    mask_no_nan = df_sub["pixelX"].notna() & df_sub["pixelY"].notna()
    
    if mask_no_nan.sum() > 0:
        # Aplicar clip solo a valores no-NaN
        df_sub.loc[mask_no_nan, "pixelX"] = np.clip(df_sub.loc[mask_no_nan, "pixelX"], 0, img_w - 1)
        df_sub.loc[mask_no_nan, "pixelY"] = np.clip(df_sub.loc[mask_no_nan, "pixelY"], 0, img_h - 1)
    
    # Identificar puntos válidos (dentro del canvas Y dentro de los límites de imagen)
    mask_canvas_valido = (df_sub["localX"] >= 0) & (df_sub["localX"] <= 1) & \
                        (df_sub["localY"] >= 0) & (df_sub["localY"] <= 1)
    
    # Verificar que pixelX y pixelY no sean NaN y estén en rangos válidos
    mask_pixel_valido = df_sub["pixelX"].notna() & df_sub["pixelY"].notna() & \
                       (df_sub["pixelX"] >= 0) & (df_sub["pixelX"] < img_w) & \
                       (df_sub["pixelY"] >= 0) & (df_sub["pixelY"] < img_h)
    
    # Todos los puntos con coordenadas válidas se consideran "procesables"
    mask_valido = mask_no_nan
    
    # Marcar estados de procesamiento
    df_sub.loc[:, "processing_status"] = "error"  # Valor por defecto
    df_sub.loc[~mask_canvas_valido, "processing_status"] = "fuera_canvas"
    df_sub.loc[mask_canvas_valido & ~mask_no_nan, "processing_status"] = "coordenadas_nan"
    df_sub.loc[mask_valido, "processing_status"] = "procesado"
    
    # Procesar TODOS los puntos que tienen coordenadas válidas (incluso si están fuera del canvas)
    if mask_valido.sum() > 0:        
        # Copiar coordenadas para pkl (ya que no escalas)
        df_sub.loc[mask_valido, "pkl_pixelX"] = df_sub.loc[mask_valido, "pixelX"]
        df_sub.loc[mask_valido, "pkl_pixelY"] = df_sub.loc[mask_valido, "pixelY"]
        
        # Extraer las clases de segmentación para puntos válidos
        unique_classes = np.unique(segmentation_array)
        total_pixels = segmentation_array.size
        
        for idx in df_sub[mask_valido].index:
            x_pkl = int(df_sub.loc[idx, "pixelX"])
            y_pkl = int(df_sub.loc[idx, "pixelY"])
            
            # Verificar que las coordenadas estén dentro del array
            if 0 <= y_pkl < segmentation_array.shape[0] and 0 <= x_pkl < segmentation_array.shape[1]:
                class_id = segmentation_array[y_pkl, x_pkl]
                df_sub.loc[idx, "class_id"] = class_id
                df_sub.loc[idx, "total_classes_in_image"] = len(unique_classes)
                
                # Calcular ratio
                class_pixels = np.sum(segmentation_array == class_id)
                df_sub.loc[idx, "class_ratio"] = class_pixels / total_pixels
            else:
                df_sub.loc[idx, "processing_status"] = "coordenadas_invalidas"
    
    return df_sub

# 1) Definir rutas y parámetros globales
ruta_all_project = "ALL_PROJECT.csv"
carpeta_seg_pkl   = "images/datos_seg" # "C:\Users\vdela\Documents\EYETRACKING\ANOTACIONES\datos_seg"
canvas_pos   = (0.0, 1.1)
canvas_size  = (0.8, 0.6)
image_pixels = (800, 600)
pkl_shape = (300, 400)
df_all = pd.read_csv(ruta_all_project)
print(f"DataFrame original: {len(df_all)} filas")

# 3) Lista para acumular cada DataFrame procesado
lista_dfs = []

# 4) Iterar por cada participante único
for participante in df_all["participante"].unique():
    print(f"Procesando participante: {participante}")
    df_part = df_all[df_all["participante"] == participante]
    for image_name in df_part["ImageName"].unique():
        print(f"  Procesando imagen: {image_name}")
        df_sub = procesar_un_subconjunto_pickle_completo(
            df_eye=df_all,
            participante=participante,
            image_name=image_name,
            carpeta_segmentaciones_pkl=carpeta_seg_pkl,
            canvas_pos=canvas_pos,
            canvas_size=canvas_size,
            image_pixels=image_pixels,
            pkl_shape=pkl_shape
        )
        if not df_sub.empty:
            lista_dfs.append(df_sub)
            print(f"    Procesadas {len(df_sub)} filas")
if lista_dfs:
    df_todo_con_clase = pd.concat(lista_dfs, ignore_index=True)
    print(f"\nDataFrame original: {len(df_all)} filas")
    print(f"DataFrame final: {len(df_todo_con_clase)} filas")
    
    # Análisis de estados de procesamiento
    print("\n=== ANÁLISIS DE PROCESAMIENTO ===")
    status_counts = df_todo_con_clase["processing_status"].value_counts()
    print("Estados de procesamiento:")
    for status, count in status_counts.items():
        percentage = (count / len(df_todo_con_clase)) * 100
        print(f"  {status}: {count} filas ({percentage:.2f}%)")
    
    # Análisis de clases (solo para filas procesadas exitosamente)
    filas_procesadas = df_todo_con_clase[df_todo_con_clase["processing_status"] == "procesado"]
    if len(filas_procesadas) > 0:
        print(f"\nFilas procesadas exitosamente: {len(filas_procesadas)}")
        print("Clases únicas encontradas:", sorted(filas_procesadas["class_id"].dropna().unique()))
        
        print("\nDistribución de clases:")
        class_counts = filas_procesadas["class_id"].value_counts().sort_index()
        for class_id, count in class_counts.items():
            percentage = (count / len(filas_procesadas)) * 100
            print(f"  Clase {class_id}: {count} puntos ({percentage:.2f}%)")
    
else:
    print("No se procesaron datos")

# Verificación final
if 'df_todo_con_clase' in locals():
    print(f"\n=== VERIFICACIÓN FINAL ===")
    print(f"Filas originales: {len(df_all)}")
    print(f"Filas finales: {len(df_todo_con_clase)}")
    print(f"Diferencia: {len(df_all) - len(df_todo_con_clase)}")
    
    if len(df_all) != len(df_todo_con_clase):
        print("ADVERTENCIA: Se perdieron filas durante el procesamiento")
        print("Posibles causas:")
        print("- Combinaciones participante/imagen que no existen en df_all")
        print("- Errores en la lógica de filtrado")
        
        # Verificar combinaciones únicas
        combinaciones_originales = set(zip(df_all["participante"], df_all["ImageName"]))
        combinaciones_procesadas = set(zip(df_todo_con_clase["participante"], df_todo_con_clase["ImageName"]))
        combinaciones_perdidas = combinaciones_originales - combinaciones_procesadas
        
        if combinaciones_perdidas:
            print(f"Combinaciones perdidas: {len(combinaciones_perdidas)}")
            for combo in list(combinaciones_perdidas)[:5]:  # Mostrar solo las primeras 5
                print(f"  Participante {combo[0]}, Imagen {combo[1]}")
    else:
        print("Perfecto: Se mantuvieron todas las filas originales")

DataFrame original: 869600 filas
Procesando participante: 1
  Procesando imagen: 3
    Procesadas 414 filas
  Procesando imagen: 4
    Procesadas 450 filas
  Procesando imagen: 5
    Procesadas 434 filas
  Procesando imagen: 7
    Procesadas 450 filas
  Procesando imagen: 8
    Procesadas 804 filas
  Procesando imagen: 9
    Procesadas 900 filas
  Procesando imagen: 30
    Procesadas 826 filas
  Procesando imagen: 32
    Procesadas 899 filas
  Procesando imagen: 33
    Procesadas 885 filas
  Procesando imagen: 36
    Procesadas 900 filas
  Procesando imagen: 39
    Procesadas 900 filas
  Procesando imagen: 41
    Procesadas 901 filas
  Procesando imagen: 43
    Procesadas 860 filas
  Procesando imagen: 47
    Procesadas 875 filas
  Procesando imagen: 48
    Procesadas 901 filas
  Procesando imagen: 50
    Procesadas 857 filas
  Procesando imagen: 51
    Procesadas 881 filas
  Procesando imagen: 56
    Procesadas 855 filas
  Procesando imagen: 57
    Procesadas 899 filas
  Procesando im

In [56]:
df_todo_con_clase

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
0,7.984,0,3,0.0052,1.0046,1.9762,1,0.006500,-0.159000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
1,8.017,0,3,0.0052,1.0060,1.9764,1,0.006500,-0.156667,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
2,8.052,0,3,0.0053,1.0070,1.9766,1,0.006625,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
3,8.084,0,3,0.0054,1.0070,1.9768,1,0.006750,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
4,8.117,0,3,0.0056,1.0059,1.9766,1,0.007000,-0.156833,6,0,6.0,0.0,3.0,9.0,0.081675,procesado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869595,1004.571,49,149,0.0034,1.0953,1.9938,30,0.004250,-0.007833,3,0,3.0,0.0,87.0,10.0,0.051433,procesado
869596,1004.605,49,149,0.0064,1.0952,1.9939,30,0.008000,-0.008000,6,0,6.0,0.0,87.0,10.0,0.051433,procesado
869597,1004.637,49,149,0.0093,1.0954,1.9939,30,0.011625,-0.007667,9,0,9.0,0.0,87.0,10.0,0.051433,procesado
869598,1004.671,49,149,0.0122,1.0956,1.9940,30,0.015250,-0.007333,12,0,12.0,0.0,87.0,10.0,0.051433,procesado


In [83]:
df_todo_con_clase[(df_todo_con_clase["ImageName"] == 1)& (df_todo_con_clase["ImageIndex"] == 0) & (df_todo_con_clase["Time"]== 9.224)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
129621,9.224,0,1,0.0307,1.1099,1.997,5,0.038375,0.0165,31,10,31.0,10.0,3.0,7.0,0.205133,procesado


In [65]:
df_todo_con_clase[(df_todo_con_clase["ImageName"] == 56) & (df_todo_con_clase["ImageIndex"] == 17) & (df_todo_con_clase["participante"] == 1)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
13137,347.518,17,56,-0.1010,1.2045,2.0154,1,-0.126250,0.174167,0,104,0.0,104.0,5.0,10.0,0.317442,procesado
13138,347.534,17,56,-0.1010,1.2044,2.0154,1,-0.126250,0.174000,0,104,0.0,104.0,5.0,10.0,0.317442,procesado
13139,347.551,17,56,-0.1010,1.2043,2.0154,1,-0.126250,0.173833,0,104,0.0,104.0,5.0,10.0,0.317442,procesado
13140,347.568,17,56,-0.1009,1.2041,2.0153,1,-0.126125,0.173500,0,104,0.0,104.0,5.0,10.0,0.317442,procesado
13141,347.584,17,56,-0.1008,1.2040,2.0153,1,-0.126000,0.173333,0,104,0.0,104.0,5.0,10.0,0.317442,procesado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13987,362.451,17,56,0.1135,1.0547,1.9862,1,0.141875,-0.075500,114,0,114.0,0.0,3.0,10.0,0.258492,procesado
13988,362.468,17,56,0.1134,1.0546,1.9862,1,0.141750,-0.075667,113,0,113.0,0.0,3.0,10.0,0.258492,procesado
13989,362.484,17,56,0.1133,1.0545,1.9862,1,0.141625,-0.075833,113,0,113.0,0.0,3.0,10.0,0.258492,procesado
13990,362.501,17,56,0.1133,1.0544,1.9862,1,0.141625,-0.076000,113,0,113.0,0.0,3.0,10.0,0.258492,procesado


In [ ]:
#aqui

In [85]:
df_todo_con_clase[(df_todo_con_clase["ImageName"] == 50) & (df_todo_con_clase["ImageIndex"] == 17) & (df_todo_con_clase["participante"] == 28) & (df_todo_con_clase["X"] == 0.1085)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
783656,348.619,17,50,0.1085,1.0857,1.9923,28,0.135625,-0.023833,108,0,108.0,0.0,5.0,9.0,0.290167,procesado


In [67]:
df_todo_con_clase.head()

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
0,7.984,0,3,0.0052,1.0046,1.9762,1,0.006500,-0.159000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
1,8.017,0,3,0.0052,1.0060,1.9764,1,0.006500,-0.156667,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
2,8.052,0,3,0.0053,1.0070,1.9766,1,0.006625,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
3,8.084,0,3,0.0054,1.0070,1.9768,1,0.006750,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
4,8.117,0,3,0.0056,1.0059,1.9766,1,0.007000,-0.156833,6,0,6.0,0.0,3.0,9.0,0.081675,procesado


In [63]:
df.shape, df_todo_con_clase.shape, PROJECT.shape

((127876, 16), (869600, 17), (869600, 7))

In [ ]:
#aquiAQUI
# 1) Definir rutas y parámetros globales
# ruta_all_project = "/Users/vdela/Documents/EYETRACKING/ALL_PROJECT.csv"
# carpeta_seg_pkl   = "/Users/vdela/Documents/EYETRACKING/ANOTACIONES/datos_seg" # "C:\Users\vdela\Documents\EYETRACKING\ANOTACIONES\datos_seg"

ruta_all_project = "ALL_PROJECT.csv"
carpeta_seg_pkl   = "images/datos_seg"  # Carpeta con archivos .pkl

canvas_pos   = (0.0, 1.1)
canvas_size  = (0.8, 0.6)
image_pixels = (800, 600)
pkl_shape = (300, 400)
df_all = pd.read_csv(ruta_all_project)
print(f"DataFrame original: {len(df_all)} filas")

# 3) Lista para acumular cada DataFrame procesado
lista_dfs = []

# 4) Iterar por cada participante único
for participante in df_all["participante"].unique():
    print(f"Procesando participante: {participante}")
    df_part = df_all[df_all["participante"] == participante]
    for image_name in df_part["ImageName"].unique():
        print(f"  Procesando imagen: {image_name}")
        df_sub = procesar_un_subconjunto_pickle_completo(
            df_eye=df_all,
            participante=participante,
            image_name=image_name,
            carpeta_segmentaciones_pkl=carpeta_seg_pkl,
            canvas_pos=canvas_pos,
            canvas_size=canvas_size,
            image_pixels=image_pixels,
            pkl_shape=pkl_shape
        )
        if not df_sub.empty:
            lista_dfs.append(df_sub)
            print(f"    Procesadas {len(df_sub)} filas")
if lista_dfs:
    df_todo_con_clase1 = pd.concat(lista_dfs, ignore_index=True)
    print(f"\nDataFrame original: {len(df_all)} filas")
    print(f"DataFrame final: {len(df_todo_con_clase1)} filas")
    
    # Análisis de estados de procesamiento
    print("\n=== ANÁLISIS DE PROCESAMIENTO ===")
    status_counts = df_todo_con_clase1["processing_status"].value_counts()
    print("Estados de procesamiento:")
    for status, count in status_counts.items():
        percentage = (count / len(df_todo_con_clase1)) * 100
        print(f"  {status}: {count} filas ({percentage:.2f}%)")
    
    # Análisis de clases (solo para filas procesadas exitosamente)
    filas_procesadas = df_todo_con_clase1[df_todo_con_clase1["processing_status"] == "procesado"]
    if len(filas_procesadas) > 0:
        print(f"\nFilas procesadas exitosamente: {len(filas_procesadas)}")
        print("Clases únicas encontradas:", sorted(filas_procesadas["class_id"].dropna().unique()))
        
        print("\nDistribución de clases:")
        class_counts = filas_procesadas["class_id"].value_counts().sort_index()
        for class_id, count in class_counts.items():
            percentage = (count / len(filas_procesadas)) * 100
            print(f"  Clase {class_id}: {count} puntos ({percentage:.2f}%)")
    
else:
    print("No se procesaron datos")

# Verificación final
if 'df_todo_con_clase1' in locals():
    print(f"\n=== VERIFICACIÓN FINAL ===")
    print(f"Filas originales: {len(df_all)}")
    print(f"Filas finales: {len(df_todo_con_clase1)}")
    print(f"Diferencia: {len(df_all) - len(df_todo_con_clase1)}")
    
    if len(df_all) != len(df_todo_con_clase1):
        print("ADVERTENCIA: Se perdieron filas durante el procesamiento")
        print("Posibles causas:")
        print("- Combinaciones participante/imagen que no existen en df_all")
        print("- Errores en la lógica de filtrado")
        
        # Verificar combinaciones únicas
        combinaciones_originales = set(zip(df_all["participante"], df_all["ImageName"]))
        combinaciones_procesadas = set(zip(df_todo_con_clase1["participante"], df_todo_con_clase1["ImageName"]))
        combinaciones_perdidas = combinaciones_originales - combinaciones_procesadas
        
        if combinaciones_perdidas:
            print(f"Combinaciones perdidas: {len(combinaciones_perdidas)}")
            for combo in list(combinaciones_perdidas)[:5]:  # Mostrar solo las primeras 5
                print(f"  Participante {combo[0]}, Imagen {combo[1]}")
    else:
        print(" Perfecto: Se mantuvieron todas las filas originales")

DataFrame original: 869600 filas
Procesando participante: 1
  Procesando imagen: 3
    Procesadas 414 filas
  Procesando imagen: 4
    Procesadas 450 filas
  Procesando imagen: 5
    Procesadas 434 filas
  Procesando imagen: 7
    Procesadas 450 filas
  Procesando imagen: 8
    Procesadas 804 filas
  Procesando imagen: 9
    Procesadas 900 filas
  Procesando imagen: 30
    Procesadas 826 filas
  Procesando imagen: 32
    Procesadas 899 filas
  Procesando imagen: 33
    Procesadas 885 filas
  Procesando imagen: 36
    Procesadas 900 filas
  Procesando imagen: 39
    Procesadas 900 filas
  Procesando imagen: 41
    Procesadas 901 filas
  Procesando imagen: 43
    Procesadas 860 filas
  Procesando imagen: 47
    Procesadas 875 filas
  Procesando imagen: 48
    Procesadas 901 filas
  Procesando imagen: 50
    Procesadas 857 filas
  Procesando imagen: 51
    Procesadas 881 filas
  Procesando imagen: 56
    Procesadas 855 filas
  Procesando imagen: 57
    Procesadas 899 filas
  Procesando im

In [88]:
df_todo_con_clase1

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
0,7.984,0,3,0.0052,1.0046,1.9762,1,0.006500,-0.159000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
1,8.017,0,3,0.0052,1.0060,1.9764,1,0.006500,-0.156667,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
2,8.052,0,3,0.0053,1.0070,1.9766,1,0.006625,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
3,8.084,0,3,0.0054,1.0070,1.9768,1,0.006750,-0.155000,5,0,5.0,0.0,3.0,9.0,0.081675,procesado
4,8.117,0,3,0.0056,1.0059,1.9766,1,0.007000,-0.156833,6,0,6.0,0.0,3.0,9.0,0.081675,procesado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869595,1004.571,49,149,0.0034,1.0953,1.9938,30,0.004250,-0.007833,3,0,3.0,0.0,87.0,10.0,0.051433,procesado
869596,1004.605,49,149,0.0064,1.0952,1.9939,30,0.008000,-0.008000,6,0,6.0,0.0,87.0,10.0,0.051433,procesado
869597,1004.637,49,149,0.0093,1.0954,1.9939,30,0.011625,-0.007667,9,0,9.0,0.0,87.0,10.0,0.051433,procesado
869598,1004.671,49,149,0.0122,1.0956,1.9940,30,0.015250,-0.007333,12,0,12.0,0.0,87.0,10.0,0.051433,procesado


In [91]:
df_todo_con_clase1[(df_todo_con_clase1["ImageName"] == 50) & (df_todo_con_clase1["ImageIndex"] == 17) & (df_todo_con_clase1["participante"] == 28) & (df_todo_con_clase1["X"] == 0.1085)]

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio,processing_status
783656,348.619,17,50,0.1085,1.0857,1.9923,28,0.135625,-0.023833,108,0,108.0,0.0,5.0,9.0,0.290167,procesado


In [105]:
df_todo_con_clase_final= pd.read_csv("df_todo_con_clase_final.csv")

In [107]:
df_todo_con_clase_final.head()

,Unnamed: 0,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,pkl_pixelX,pkl_pixelY,class_id,total_classes_in_image,class_ratio
0,0,14.184,0,3,0.0012,1.1145,1.9977,1,0.001500,0.024167,1,14,0,7,3,9,0.081675
1,1,14.224,0,3,0.0078,1.1132,1.9975,1,0.009750,0.022000,7,13,3,6,3,9,0.081675
2,2,14.951,0,3,0.1293,1.1397,2.0027,1,0.161625,0.066167,129,39,64,19,5,9,0.328425
3,3,14.984,0,3,0.1293,1.1396,2.0027,1,0.161625,0.066000,129,39,64,19,5,9,0.328425
4,4,15.017,0,3,0.1297,1.1391,2.0026,1,0.162125,0.065167,129,39,64,19,5,9,0.328425


In [101]:
df_todo_con_clase_ratio= pd.read_csv("df_todo_con_clase_ratio.csv")

In [109]:
df_todo_con_clase_ratio.head()

,Unnamed: 0,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,class_id,class_name,ratio
0,0,7.984,0,3,0.0052,1.0046,1.9762,1,0.0052,-0.0954,405.2,204.6,7.0,road;route,29.615
1,1,8.017,0,3,0.0052,1.0060,1.9764,1,0.0052,-0.0940,405.2,206.0,7.0,road;route,29.615
2,2,8.052,0,3,0.0053,1.0070,1.9766,1,0.0053,-0.0930,405.3,207.0,7.0,road;route,29.615
3,3,8.084,0,3,0.0054,1.0070,1.9768,1,0.0054,-0.0930,405.4,207.0,7.0,road;route,29.615
4,4,8.117,0,3,0.0056,1.0059,1.9766,1,0.0056,-0.0941,405.6,205.9,7.0,road;route,29.615


In [113]:
df_todo_con_clase_ratio.shape

(869600, 15)

In [ ]:
##############

In [17]:
import pandas as pd
import numpy as np
import os
from PIL import Image


In [18]:
def procesar_un_subconjunto(
    df_eye: pd.DataFrame,
    participante: int,
    image_name: int,
    carpeta_segmentaciones: str,
    carpeta_segment_img: str,
    canvas_pos=(0.0, 1.1),
    canvas_size=(0.8, 0.6),
    image_pixels=(800, 600),
    tolerancia_rgb: int = 2
) -> pd.DataFrame:
    """
    A partir de df_eye (ALL_PROJECT completo), filtra el subconjunto donde 
    'participante' y 'ImageName' coincidan. Calcula localX/localY → pixelX/pixelY,
    asigna class_id y class_name según la máscara segmentada, y añade la columna
    'ratio' que proviene del CSV de segmentación. Devuelve ese DataFrame reducido
    con las columnas adicionales ['localX','localY','pixelX','pixelY','class_id',
    'class_name','ratio'].

    Parámetros:
    - df_eye: DataFrame completo de ALL_PROJECT.csv
    - participante: entero, ID del participante
    - image_name: entero o string, ID de la imagen (ej. 0, 1, 2…)
    - carpeta_segmentaciones: ruta donde está el CSV de segmentación (p.ej. "0.csv")
    - carpeta_segment_img: ruta donde está la máscara coloreada (p.ej. "0.JPEG")
    - canvas_pos, canvas_size, image_pixels, tolerancia_rgb: igual que antes
    """

    # 1) Filtrar las filas correspondientes a ese (participante, image_name)
    df0 = df_eye.loc[
        (df_eye["participante"] == participante) &
        (df_eye["ImageName"]    == image_name)
    ].copy()

    if df0.empty:
        # No hay filas para este par; devolvemos un DataFrame vacío con las columnas esperadas
        columnas = list(df_eye.columns) + [
            "localX", "localY", "pixelX", "pixelY",
            "class_id", "class_name", "ratio"
        ]
        return pd.DataFrame(columns=columnas)

    # 2) World → local
    cx, cy = canvas_pos
    df0["localX"] = df0["X"] - cx
    df0["localY"] = df0["Y"] - cy

    # 3) Escalado a píxeles
    world_w, world_h = canvas_size
    px_w, px_h       = image_pixels
    scale_x = px_w / world_w
    scale_y = px_h / world_h

    df0["pixelX"] = df0["localX"] * scale_x + px_w / 2
    df0["pixelY"] = df0["localY"] * scale_y + px_h / 2

    # 4) Preparar columnas para la asignación
    df0["class_id"] = np.nan
    df0["class_name"] = ""   # campo para el nombre de la clase
    df0["ratio"] = np.nan    # aquí guardaremos el ratio de la clase

    # 5) Cargar CSV y máscara de segmentación si existen
    ruta_csv_seg = os.path.join(carpeta_segmentaciones, f"{image_name}.csv")
    ruta_img_seg = os.path.join(carpeta_segment_img,    f"{image_name}.JPEG")

    if os.path.isfile(ruta_csv_seg) and os.path.isfile(ruta_img_seg):
        # 5.1) Leer CSV de segmentación
        df_seg = pd.read_csv(ruta_csv_seg, sep=";")
        # Esperamos columnas: ['class_id','RGB_color','class_name','ratio']

        # Convertir la columna RGB_color "(r, g, b)" → tupla (r, g, b)
        def str_a_tupla(rgb_str: str):
            s = rgb_str.strip().lstrip("(").rstrip(")")
            comps = [c.strip() for c in s.split(",")]
            return tuple(int(c) for c in comps)

        df_seg["RGB_tuple"] = df_seg["RGB_color"].apply(str_a_tupla)

        # Construir diccionarios de mapeo:
        #   mapa_rgb_a_clase:  (r,g,b) → class_id
        #   mapa_rgb_a_nombre: (r,g,b) → class_name
        mapa_rgb_a_clase  = { tup: int(cid)   for cid,  tup in zip(df_seg["class_id"],  df_seg["RGB_tuple"]) }
        mapa_rgb_a_nombre = { tup: cname      for cname,tup in zip(df_seg["class_name"], df_seg["RGB_tuple"]) }
        # Diccionario para ratio, usando class_name como llave:
        mapa_nombre_a_ratio = { cname: r for cname, r in zip(df_seg["class_name"], df_seg["ratio"]) }

        # 5.2) Cargar máscara coloreada y redimensionar a (px_w, px_h) con NEAREST
        img_seg_original = Image.open(ruta_img_seg).convert("RGB")
        img_seg = img_seg_original.resize((px_w, px_h), resample=Image.NEAREST)
        seg_w, seg_h = img_seg.size  # deberían ser (800, 600)

        # 5.3) Función para asignar (class_id, class_name) con tolerancia
        def obtener_clase_con_tolerancia(color_ext, mapa_clase, mapa_nombre, umbral):
            # 5.3.1) Coincidencia exacta:
            if color_ext in mapa_clase:
                return mapa_clase[color_ext], mapa_nombre[color_ext]

            # 5.3.2) Búsqueda dentro del umbral:
            r1, g1, b1 = color_ext
            for (r0, g0, b0), cid in mapa_clase.items():
                if (abs(r1 - r0) <= umbral and
                    abs(g1 - g0) <= umbral and
                    abs(b1 - b0) <= umbral):
                    return cid, mapa_nombre[(r0, g0, b0)]

            # 5.3.3) Si no encuentra, devolvemos (NaN, "")
            return np.nan, ""

        # 6) Iterar sobre cada fila de df0 y asignar class_id, class_name y ratio
        for idx in df0.index:
            px = df0.at[idx, "pixelX"]
            py = df0.at[idx, "pixelY"]

            # 6.1) Convertir a coordenadas (x_seg, y_seg) medidas desde arriba
            x_seg = int(round(px))
            y_seg = int(round(px_h - py))

            # Clamp a los límites de la máscara
            x_seg = max(0, min(x_seg, seg_w - 1))
            y_seg = max(0, min(y_seg, seg_h - 1))

            # 6.2) Extraer color y buscar clase
            color = img_seg.getpixel((x_seg, y_seg))  # (r, g, b)
            cid, cname = obtener_clase_con_tolerancia(
                color, mapa_rgb_a_clase, mapa_rgb_a_nombre, tolerancia_rgb
            )
            df0.at[idx, "class_id"] = cid
            df0.at[idx, "class_name"] = cname

            # 6.3) Asignar ratio usando el diccionario por nombre de clase
            df0.at[idx, "ratio"] = mapa_nombre_a_ratio.get(cname, np.nan)

    # Si no existe CSV o máscara, todas las filas mantendrán class_id=NaN, class_name="" y ratio=NaN
    return df0

In [19]:
# 1) Definir rutas y parámetros globales
# ruta_all_project = "/Users/vdela/Documents/EYETRACKING/ALL_PROJECT.csv"
# carpeta_seg_csv   = "/Users/vdela/Documents/EYETRACKING/images/datos_seg"
# carpeta_seg_img   = "/Users/vdela/Documents/EYETRACKING/images/images_seg"

ruta_all_project = "ALL_PROJECT.csv"
carpeta_seg_csv   = "images/datos_seg"  # Carpeta con archivos .pkl
carpeta_seg_img   = "images/images_seg"

canvas_pos   = (0.0, 1.1)
canvas_size  = (0.8, 0.6)
image_pixels = (800, 600)
tolerancia_rgb = 5

# 2) Leer todo el ALL_PROJECT.csv
df_all = pd.read_csv(ruta_all_project)

# 3) Lista para acumular cada DataFrame procesado
lista_dfs = []

# 4) Iterar por cada participante único
for participante in df_all["participante"].unique():
    # Filtrar las filas de ese participante
    df_part = df_all[ df_all["participante"] == participante ]

    # 5) Para cada ImageName dentro de este participante:
    for image_name in df_part["ImageName"].unique():
        # 5.1) Llamar a la función para procesar ese bloque
        df_sub = procesar_un_subconjunto(
            df_eye=df_all,
            participante=participante,
            image_name=image_name,
            carpeta_segmentaciones=carpeta_seg_csv,
            carpeta_segment_img=carpeta_seg_img,
            canvas_pos=canvas_pos,
            canvas_size=canvas_size,
            image_pixels=image_pixels,
            tolerancia_rgb=tolerancia_rgb
        )

        # 5.2) Si df_sub no está vacío lo guardamos en la lista
        if not df_sub.empty:
            lista_dfs.append(df_sub)


# 6) Concatenar todos los DataFrames parciales
if lista_dfs:
    df_todo_con_clase = pd.concat(lista_dfs, ignore_index=True)
else:
    # Si no se procesó nada, devolvemos un DataFrame vacío con las columnas esperadas
    columnas_finales = list(df_all.columns) + ["localX","localY","pixelX","pixelY","class_id","class_name","ratio"]
    df_todo_con_clase = pd.DataFrame(columns=columnas_finales)


In [20]:
df_todo_con_clase

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,class_id,class_name,ratio
0,7.984,0,3,0.0052,1.0046,1.9762,1,0.0052,-0.0954,405.2,204.6,7.0,road;route,29.615
1,8.017,0,3,0.0052,1.0060,1.9764,1,0.0052,-0.0940,405.2,206.0,7.0,road;route,29.615
2,8.052,0,3,0.0053,1.0070,1.9766,1,0.0053,-0.0930,405.3,207.0,7.0,road;route,29.615
3,8.084,0,3,0.0054,1.0070,1.9768,1,0.0054,-0.0930,405.4,207.0,7.0,road;route,29.615
4,8.117,0,3,0.0056,1.0059,1.9766,1,0.0056,-0.0941,405.6,205.9,7.0,road;route,29.615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869595,1004.571,49,149,0.0034,1.0953,1.9938,30,0.0034,-0.0047,403.4,295.3,NaN,,NaN
869596,1004.605,49,149,0.0064,1.0952,1.9939,30,0.0064,-0.0048,406.4,295.2,NaN,,NaN
869597,1004.637,49,149,0.0093,1.0954,1.9939,30,0.0093,-0.0046,409.3,295.4,NaN,,NaN
869598,1004.671,49,149,0.0122,1.0956,1.9940,30,0.0122,-0.0044,412.2,295.6,NaN,,NaN


In [50]:
def rgb_str_to_hex(rgb_str):
    """Convierte '(140, 140, 140)' a '#8C8C8C'"""
    try:
        # Limpiar paréntesis y dividir
        s = rgb_str.strip().lstrip("(").rstrip(")")
        parts = [int(c) for c in s.split(",")]
        # Formatear a Hex
        return '#{:02x}{:02x}{:02x}'.format(*parts).upper()
    except:
        return "#000000" # Fallback en caso de error

def extract_main_class(full_name):
    """Convierte 'road;route' a 'road'"""
    if pd.isna(full_name): return ""
    # Tomar solo la primera parte antes del punto y coma
    return str(full_name).split(';')[0].strip()

In [59]:
def procesar_un_subconjunto_optimizado(
    df_eye: pd.DataFrame,
    participante: int,
    image_name: int,
    carpeta_segmentaciones_pkl: str, 
    carpeta_csv_mapping: str,        
    canvas_pos=(0.0, 1.1),
    canvas_size=(0.8, 0.6),
    image_pixels=(800, 600)
) -> pd.DataFrame:
    
    # 1) Filtrar vectorizadamente
    df0 = df_eye[
        (df_eye["participante"] == participante) & 
        (df_eye["ImageName"] == image_name)
    ].copy()

    if df0.empty:
        return pd.DataFrame()

    # 2) Transformación Geométrica
    cx, cy = canvas_pos
    world_w, world_h = canvas_size
    px_w, px_h = image_pixels
    
    scale_x = px_w / world_w
    scale_y = px_h / world_h

    df0["pixelX"] = (df0["X"] - cx) * scale_x + (px_w / 2)
    pixel_y_raw = (df0["Y"] - cy) * scale_y + (px_h / 2)
    #df0["pixelY"] = px_h - pixel_y_raw 
    df0["pixelY"] = pixel_y_raw
    # 3) Indices para buscar en la matriz
    x_idxs = np.clip(df0["pixelX"].fillna(-1).astype(int), 0, px_w - 1)
    y_idxs = np.clip(df0["pixelY"].fillna(-1).astype(int), 0, px_h - 1)

    # 4) Cargar Archivos
    ruta_pkl = os.path.join(carpeta_segmentaciones_pkl, f"{image_name}.pkl")
    ruta_csv = os.path.join(carpeta_csv_mapping, f"{image_name}.csv")

    if os.path.isfile(ruta_pkl) and os.path.isfile(ruta_csv):
        try:
            # --- CARGAR MÁSCARA ---
            mask_array = joblib.load(ruta_pkl) 
            
            if mask_array.shape != (px_h, px_w):
                mask_img = Image.fromarray(mask_array.astype('int32'))
                mask_img = mask_img.resize((px_w, px_h), resample=Image.NEAREST)
                mask_array = np.array(mask_img)

            # Asignar ID
            df0["class_id"] = mask_array[y_idxs, x_idxs]

            # --- CARGAR CSV Y CREAR NUEVAS COLUMNAS ---
            df_mapping = pd.read_csv(ruta_csv, sep=";")
            
            # A) Crear columna hex_color desde RGB_color
            if "RGB_color" in df_mapping.columns:
                df_mapping["hex_color"] = df_mapping["RGB_color"].apply(rgb_str_to_hex)
            else:
                df_mapping["hex_color"] = "#000000"

            # B) Crear columna main_class desde class_name
            if "class_name" in df_mapping.columns:
                df_mapping["main_class"] = df_mapping["class_name"].apply(extract_main_class)
            else:
                df_mapping["main_class"] = "unknown"

            # C) Crear Diccionarios de Mapeo
            id_to_name  = dict(zip(df_mapping['class_id'], df_mapping['class_name']))
            id_to_ratio = dict(zip(df_mapping['class_id'], df_mapping['ratio']))
            id_to_hex   = dict(zip(df_mapping['class_id'], df_mapping['hex_color']))
            id_to_main  = dict(zip(df_mapping['class_id'], df_mapping['main_class']))

            # D) Mapear al DataFrame principal
            df0["class_name"] = df0["class_id"].map(id_to_name)
            df0["ratio"]      = df0["class_id"].map(id_to_ratio)
            df0["hex_color"]  = df0["class_id"].map(id_to_hex)
            df0["main_class"] = df0["class_id"].map(id_to_main)

        except Exception as e:
            print(f"Error procesando imagen {image_name}: {e}")
            # Llenar con NaNs si falla
            for col in ["class_id", "class_name", "ratio", "hex_color", "main_class"]:
                df0[col] = np.nan
    else:
        # Si no existen archivos
        for col in ["class_id", "class_name", "ratio", "hex_color", "main_class"]:
            df0[col] = np.nan

    return df0

In [60]:
# 1) Definir rutas y parámetros globales
# ruta_all_project = "/Users/vdela/Documents/EYETRACKING/ALL_PROJECT.csv"
# carpeta_seg_csv   = "/Users/vdela/Documents/EYETRACKING/images/datos_seg"
# carpeta_seg_img   = "/Users/vdela/Documents/EYETRACKING/images/images_seg"

ruta_all_project = "ALL_PROJECT.csv"
carpeta_seg_csv   = "images/datos_seg"  # Carpeta con archivos .pkl
carpeta_seg_img   = "images/images_seg"
carpeta_pkl = "images/datos_seg"
carpeta_csv = "images/datos_seg"

canvas_pos   = (0.0, 1.1)
canvas_size  = (0.8, 0.6)
image_pixels = (800, 600)

# 2) Leer todo el ALL_PROJECT.csv
df_all = pd.read_csv(ruta_all_project)

# 3) Lista para acumular cada DataFrame procesado
lista_dfs = []

# 4) Iterar por cada participante único
for participante in df_all["participante"].unique():
    # Filtrar las filas de ese participante
    df_part = df_all[ df_all["participante"] == participante ]

    # 5) Para cada ImageName dentro de este participante:
    for image_name in df_part["ImageName"].unique():
        # 5.1) Llamar a la función para procesar ese bloque
        df_sub = procesar_un_subconjunto_optimizado( 
            df_eye=df_all,
            participante=participante,
            image_name=image_name,
            carpeta_segmentaciones_pkl=carpeta_pkl,
            carpeta_csv_mapping=carpeta_seg_csv,
            #carpeta_segment_img=carpeta_seg_img,
            canvas_pos=canvas_pos,
            canvas_size=canvas_size,
            image_pixels=image_pixels,
            #tolerancia_rgb=tolerancia_rgb
        )

        # 5.2) Si df_sub no está vacío lo guardamos en la lista
        if not df_sub.empty:
            lista_dfs.append(df_sub)


# 6) Concatenar todos los DataFrames parciales
if lista_dfs:
    df_todo_con_clase = pd.concat(lista_dfs, ignore_index=True)
    print("Procesamiento completado. Filas totales:", len(df_todo_con_clase))
else:
    # Si no se procesó nada, devolvemos un DataFrame vacío con las columnas esperadas
    print("No se generaron datos.")
    columnas_finales = list(df_all.columns) + ["localX","localY","pixelX","pixelY","class_id","class_name","ratio"]
    df_todo_con_clase = pd.DataFrame(columns=columnas_finales)


Procesamiento completado. Filas totales: 869600


In [62]:
df_todo_con_clase.head()

,Time,ImageIndex,ImageName,X,Y,Z,participante,pixelX,pixelY,class_id,class_name,ratio,hex_color,main_class
0,7.984,0,3,0.0052,1.0046,1.9762,1,405.2,204.6,5,tree,32.8425,#04C803,tree
1,8.017,0,3,0.0052,1.0060,1.9764,1,405.2,206.0,5,tree,32.8425,#04C803,tree
2,8.052,0,3,0.0053,1.0070,1.9766,1,405.3,207.0,5,tree,32.8425,#04C803,tree
3,8.084,0,3,0.0054,1.0070,1.9768,1,405.4,207.0,5,tree,32.8425,#04C803,tree
4,8.117,0,3,0.0056,1.0059,1.9766,1,405.6,205.9,5,tree,32.8425,#04C803,tree


In [57]:
df_todo_con_clase.head()

,Time,ImageIndex,ImageName,X,Y,Z,participante,pixelX,pixelY,class_id,class_name,ratio,hex_color,main_class
0,7.984,0,3,0.0052,1.0046,1.9762,1,405.2,395.4,7,road;route,29.615,#8C8C8C,road
1,8.017,0,3,0.0052,1.0060,1.9764,1,405.2,394.0,7,road;route,29.615,#8C8C8C,road
2,8.052,0,3,0.0053,1.0070,1.9766,1,405.3,393.0,7,road;route,29.615,#8C8C8C,road
3,8.084,0,3,0.0054,1.0070,1.9768,1,405.4,393.0,7,road;route,29.615,#8C8C8C,road
4,8.117,0,3,0.0056,1.0059,1.9766,1,405.6,394.1,7,road;route,29.615,#8C8C8C,road


In [63]:
df_todo_con_clase.to_csv('df_final1.csv') 

In [39]:
df_todo_con_clase

,Time,ImageIndex,ImageName,X,Y,Z,participante,pixelX,pixelY,class_id,class_name,ratio
0,7.984,0,3,0.0052,1.0046,1.9762,1,405.2,204.6,5,tree,32.842500
1,8.017,0,3,0.0052,1.0060,1.9764,1,405.2,206.0,5,tree,32.842500
2,8.052,0,3,0.0053,1.0070,1.9766,1,405.3,207.0,5,tree,32.842500
3,8.084,0,3,0.0054,1.0070,1.9768,1,405.4,207.0,5,tree,32.842500
4,8.117,0,3,0.0056,1.0059,1.9766,1,405.6,205.9,5,tree,32.842500
...,...,...,...,...,...,...,...,...,...,...,...,...
869595,1004.571,49,149,0.0034,1.0953,1.9938,30,403.4,295.3,2,building;edifice,19.054167
869596,1004.605,49,149,0.0064,1.0952,1.9939,30,406.4,295.2,2,building;edifice,19.054167
869597,1004.637,49,149,0.0093,1.0954,1.9939,30,409.3,295.4,2,building;edifice,19.054167
869598,1004.671,49,149,0.0122,1.0956,1.9940,30,412.2,295.6,2,building;edifice,19.054167


In [30]:
df_todo_con_clase

,Time,ImageIndex,ImageName,X,Y,Z,participante,localX,localY,pixelX,pixelY,class_id,class_name,ratio
0,7.984,0,3,0.0052,1.0046,1.9762,1,0.0052,-0.0954,405.2,204.6,7.0,road;route,29.615
1,8.017,0,3,0.0052,1.0060,1.9764,1,0.0052,-0.0940,405.2,206.0,7.0,road;route,29.615
2,8.052,0,3,0.0053,1.0070,1.9766,1,0.0053,-0.0930,405.3,207.0,7.0,road;route,29.615
3,8.084,0,3,0.0054,1.0070,1.9768,1,0.0054,-0.0930,405.4,207.0,7.0,road;route,29.615
4,8.117,0,3,0.0056,1.0059,1.9766,1,0.0056,-0.0941,405.6,205.9,7.0,road;route,29.615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869595,1004.571,49,149,0.0034,1.0953,1.9938,30,0.0034,-0.0047,403.4,295.3,NaN,,NaN
869596,1004.605,49,149,0.0064,1.0952,1.9939,30,0.0064,-0.0048,406.4,295.2,NaN,,NaN
869597,1004.637,49,149,0.0093,1.0954,1.9939,30,0.0093,-0.0046,409.3,295.4,NaN,,NaN
869598,1004.671,49,149,0.0122,1.0956,1.9940,30,0.0122,-0.0044,412.2,295.6,NaN,,NaN


In [27]:
df_todo_con_clase.isnull().sum()

Time                 0
ImageIndex           0
ImageName            0
X                    0
Y                    0
Z                    0
participante         0
localX               0
localY               0
pixelX               0
pixelY               0
class_id        261454
class_name           0
ratio           261454
dtype: int64

In [31]:
df_todo_con_clase.isnull().sum()

Time                 0
ImageIndex           0
ImageName            0
X                    0
Y                    0
Z                    0
participante         0
localX               0
localY               0
pixelX               0
pixelY               0
class_id        261454
class_name           0
ratio           261454
dtype: int64